In [7]:
from __future__ import division
get_ipython().magic('matplotlib inline')
import numpy as np, matplotlib.pyplot as plt, math, numpy.random as npr, pystan as ps
from pylab import plot, show, legend
from time import time
from tqdm import trange

/tmp/ipykernel_3869588/2585968010.py:2: DeprecationWarning: `magic(...)` is deprecated since IPython 0.13 (warning added in 8.1), use run_line_magic(magic_name, parameter_s).
  get_ipython().magic('matplotlib inline')


In [8]:
sm = ps.StanModel(file="model.stan",verbose=True)

INFO:pystan:COMPILING THE C++ CODE FOR MODEL anon_model_8c87c6041a95014f13b44d90acde3cdc NOW.
INFO:pystan:OS: linux, Python: 3.8.20 (default, Oct  3 2024, 15:24:27) 
[GCC 11.2.0], Cython 3.2.3


Compiling /tmp/pystan_s039qoz3/stanfit4anon_model_8c87c6041a95014f13b44d90acde3cdc_3272795809907284575.pyx because it changed.
[1/1] Cythonizing /tmp/pystan_s039qoz3/stanfit4anon_model_8c87c6041a95014f13b44d90acde3cdc_3272795809907284575.pyx


INFO:root:building 'stanfit4anon_model_8c87c6041a95014f13b44d90acde3cdc_3272795809907284575' extension
INFO:root:creating /tmp/pystan_s039qoz3/tmp/pystan_s039qoz3
INFO:root:g++ -pthread -B /data/student/miniconda3/envs/dcbats-legacy/compiler_compat -Wl,--sysroot=/ -Wsign-compare -DNDEBUG -g -fwrapv -O3 -Wall -Wstrict-prototypes -fPIC -DBOOST_RESULT_OF_USE_TR1 -DBOOST_NO_DECLTYPE -DBOOST_DISABLE_ASSERTS -I/tmp/pystan_s039qoz3 -I/data/student/miniconda3/envs/dcbats-legacy/lib/python3.8/site-packages/pystan -I/data/student/miniconda3/envs/dcbats-legacy/lib/python3.8/site-packages/pystan/stan/src -I/data/student/miniconda3/envs/dcbats-legacy/lib/python3.8/site-packages/pystan/stan/lib/stan_math -I/data/student/miniconda3/envs/dcbats-legacy/lib/python3.8/site-packages/pystan/stan/lib/stan_math/lib/eigen_3.3.3 -I/data/student/miniconda3/envs/dcbats-legacy/lib/python3.8/site-packages/pystan/stan/lib/stan_math/lib/boost_1.69.0 -I/data/student/miniconda3/envs/dcbats-legacy/lib/python3.8/site-pa

KeyboardInterrupt: 

In [ ]:
T = 10**5
p = 1
alpha = 0
phi = [1.942, -0.943]
sigmasq = 1

In [ ]:
# y = np.zeros(T)
# X = npr.randn(T,p)
# beta = npr.randn(p)/10
# epsilon = np.zeros(T)
# epsilon[:2] = npr.randn(2)
# for t in np.arange(2,T) :
#     epsilon[t] = sum(phi*(epsilon[t-2:t][::-1])) + np.sqrt(sigmasq)*npr.randn()
    
# for t in range(T) : y[t] = alpha + sum(beta*X[t]) + epsilon[t]
# np.save('/xtmp/PIE_time_series_data/ar(2)_errors/y.npy', y)
# np.save('/xtmp/PIE_time_series_data/ar(2)_errors/X.npy', X)
# np.save('/xtmp/PIE_time_series_data/ar(2)_errors/beta.npy', beta)

y = np.load('/data/student/Zixuan-UTS/DC-BATS/data/LM/y.npy')[1:]
X = np.load('/data/student/Zixuan-UTS/DC-BATS/data/LM/X.npy')[1:]
#beta = np.load('/data/student/Zixuan-UTS/DC-BATS/data/LM/beta.npy')

In [ ]:
plt.plot(y)

### Sanity check using one chunk:

In [ ]:
j = 1
tmin = j*10**4
tmax = (j+1)*10**4
n_chains = 10
n_iter = 10**3

#### Power = 1:

In [ ]:
m = 1
data = dict(K=tmax-tmin, p=p, m=m, y=y[tmin:tmax], X=X[tmin:tmax].transpose())
start = time()
fit = sm.sampling(data=data, thin=1, n_jobs=min(10,n_chains), chains=n_chains, init="random", iter=n_iter)
print(round((time()-start)/60,2), "minutes to run")
trace = fit.extract()

In [ ]:
alpha_CI = np.percentile(trace['alpha'], [5,95])
beta_CI = np.percentile(trace['beta'], [2.5,97.5], axis=0)
CI_contains = (beta_CI[0]<beta)*(beta<beta_CI[1])
print("Beta CI coverage =", np.mean(CI_contains)*100, "%")
print("Beta CI average length =", np.mean(np.abs(beta_CI[1]-beta_CI[0])))
phi_CI = np.percentile(trace['phi'], [2.7,97.5], axis=0)
CI_contains = (phi_CI[0]<phi)*(phi<phi_CI[1])
print(CI_contains)

In [ ]:
plt.figure(figsize=(12,4))
plt.subplot(121)
plt.plot(trace['sigmasq'])
plt.subplot(122)
plt.plot(trace['beta'][:,1])

In [ ]:
f, (a0, a1, a2) = plt.subplots(1, 3, gridspec_kw={'width_ratios': [10, 1, 0.5]}, figsize=(16,4))
a0.plot(np.arange(1,p+1), beta, "o")
for i in range(p) :
    a0.plot([i+1,i+1], beta_CI[:,i], "r-")
    a0.grid(True)
a1.plot(np.arange(1,3),phi, "o")
for i in range(len(phi)) :
    a1.plot([i+1,i+1], phi_CI[:,i], "r-")
    a1.grid(True)
a1.set_xlim([1/2,len(phi)+1/2])
a2.plot(1, alpha, "o")
a2.plot([1,1], alpha_CI, "r-")
a2.grid(True)
plt.suptitle("Credible intervals");

#### Correct power:

In [ ]:
m = T/(tmax-tmin)
data = dict(K=tmax-tmin, p=p, m=m, y=y[tmin:tmax], X=X[tmin:tmax].transpose())
start = time()
fit = sm.sampling(data=data, thin=1, n_jobs=min(10,n_chains), chains=n_chains, init="random", iter=n_iter)
print(round((time()-start)/60,2), "minutes to run")
trace = fit.extract()

In [ ]:
alpha_CI = np.percentile(trace['alpha'], [5,95])
beta_CI = np.percentile(trace['beta'], [2.5,97.5], axis=0)
CI_contains = (beta_CI[0]<beta)*(beta<beta_CI[1])
print("Beta CI coverage =", np.mean(CI_contains)*100, "%")
print("Beta CI average length =", np.mean(np.abs(beta_CI[1]-beta_CI[0])))
phi_CI = np.percentile(trace['phi'], [2.7,97.5], axis=0)
CI_contains = (phi_CI[0]<phi)*(phi<phi_CI[1])
print(CI_contains)

In [ ]:
plt.figure(figsize=(12,4))
plt.subplot(121)
plt.plot(trace['sigmasq'])
plt.subplot(122)
plt.plot(trace['beta'][:,1])

In [ ]:
f, (a0, a1, a2) = plt.subplots(1, 3, gridspec_kw={'width_ratios': [10, 1, 0.5]}, figsize=(16,4))
a0.plot(np.arange(1,p+1), beta, "o")
for i in range(p) :
    a0.plot([i+1,i+1], beta_CI[:,i], "r-")
    a0.grid(True)
a1.plot(np.arange(1,3),phi, "o")
for i in range(len(phi)) :
    a1.plot([i+1,i+1], phi_CI[:,i], "r-")
    a1.grid(True)
a1.set_xlim([1/2,len(phi)+1/2])
a2.plot(1, alpha, "o")
a2.plot([1,1], alpha_CI, "r-")
a2.grid(True)
plt.suptitle("Credible intervals");

### Divide and conquer:

In [ ]:
m = 10.
tstart = np.arange(m).astype(int)
tend = 1 + tstart
tstart *= int(T/m)
tend *= int(T/m)

n_chains = 10
n_iter = 10**3
trace_all = []
for i in range(int(m)) :
    tmin, tmax = tstart[i], tend[i]
    data = dict(K=tmax-tmin, p=p, m=m, y=y[tmin:tmax], X=X[tmin:tmax].transpose())
    fit = sm.sampling(data=data, thin=1, n_jobs=min(10,n_chains), chains=n_chains, init="random", iter=n_iter)
    trace = fit.extract()
    trace_all.append(trace)

np.save("/data/student/Zixuan-UTS/DC-BATS/data/LM/trace_all.npy", trace_all, allow_pickle=True)

In [ ]:
m = 20.
tstart = np.arange(m).astype(int)
tend = 1 + tstart
tstart *= int(T/m)
tend *= int(T/m)

n_chains = 10
n_iter = 10**3
beta_vals_2 = np.zeros((int(m),int(n_chains*n_iter/2),p))
for i in range(int(m)) :
    tmin, tmax = tstart[i], tend[i]
    data = dict(K=tmax-tmin, p=p, m=m, y=y[tmin:tmax], X=X[tmin:tmax].transpose())
    fit = sm.sampling(data=data, thin=1, n_jobs=min(10,n_chains), chains=n_chains, init="random", iter=n_iter)
    trace = fit.extract()
    beta_vals_2[i] = trace['beta'] 
np.save('/xtmp/DC-BATS_data/ar(2)_errors/beta_vals_1.npy', beta_vals_1)
np.save('/xtmp/DC-BATS_data/ar(2)_errors/beta_vals_2.npy', beta_vals_2)

In [ ]:
np.shape(beta_vals_1)

In [ ]:
beta_CI_1 = np.zeros((p,2,10))
for i in range(p) :
    beta_CI_1[i] = np.percentile(beta_vals_1[:,:,i],axis=1,q=[2.5,97.5])
beta_CI_2 = np.zeros((p,2,20))
for i in range(p) :
    beta_CI_2[i] = np.percentile(beta_vals_2[:,:,i],axis=1,q=[2.5,97.5])

In [ ]:
beta_ci_1 = np.mean(beta_CI_1,-1)
beta_ci_2 = np.mean(beta_CI_2,-1)

In [ ]:
np.shape(beta_ci_1)

In [ ]:
np.mean(beta_ci_1[:,1]-beta_ci_1[:,0])
np.mean(beta_ci_2[:,1]-beta_ci_2[:,0])

In [ ]:
print("Beta CI average length: m = 10, length =", np.mean(beta_ci_1[:,1]-beta_ci_1[:,0]), 
      "; m = 20, length =", np.mean(beta_ci_2[:,1]-beta_ci_2[:,0]))

In [ ]:
fig = plt.figure(figsize=(16,4))
plt.subplot(121)
plt.plot(np.arange(1,p+1), beta, "o")
for i in range(p) :
    plt.plot([i+1,i+1], beta_ci_1[i], "r-")
plt.grid(True)
plt.xlabel(r"$\beta$ index", fontsize=12)
plt.ylabel(r"$\beta$", fontsize=12)
plt.title(r"$m=10$")
plt.subplot(122)
plt.plot(np.arange(1,p+1), beta, "o")
for i in range(p) :
    plt.plot([i+1,i+1], beta_ci_2[i], "r-")
plt.grid(True)
plt.title(r"$m=20$")
plt.xlabel(r"$\beta$ index", fontsize=12)
plt.ylabel(r"$\beta$", fontsize=12)
plt.suptitle(r"Credible intervals for $\beta$ using divide-and-conquer", fontsize=14);

### Full posterior:

In [ ]:
data = dict(K=T, p=p, m=1, y=y, X=X.transpose())
start = time()
fit_full = sm.sampling(data=data, thin=1, n_jobs=min(10,10), chains=10, init="random", iter=5000)
print(round((time()-start)/60,2), "minutes to run")
trace_full = fit_full.extract()
#np.save('/xtmp/DC-BATS_data/ar(2)_errors/beta_full.npy', trace_full['beta'])

In [ ]:
alpha_CI = np.percentile(trace_full['alpha'], [5,95])
beta_CI = np.percentile(trace_full['beta'], [2.5,97.5], axis=0)
CI_contains = (beta_CI[0]<beta)*(beta<beta_CI[1])
print("Beta CI coverage =", np.mean(CI_contains)*100, "%")
print("Beta CI average length =", np.mean(np.abs(beta_CI[1]-beta_CI[0])))
phi_CI = np.percentile(trace_full['phi'], [2.7,97.5], axis=0)
CI_contains = (phi_CI[0]<phi)*(phi<phi_CI[1])
print(CI_contains)

In [ ]:
plt.figure(figsize=(12,4))
plt.subplot(121)
plt.plot(trace_full['sigmasq'])
plt.subplot(122)
plt.plot(trace_full['beta'][:,1])

In [ ]:
f, (a0, a1, a2) = plt.subplots(1, 3, gridspec_kw={'width_ratios': [10, 1, 0.5]}, figsize=(16,4))
a0.plot(np.arange(1,p+1), beta, "o")
for i in range(p) :
    a0.plot([i+1,i+1], beta_CI[:,i], "r-")
    a0.grid(True)
a1.plot(np.arange(1,3),phi, "o")
for i in range(len(phi)) :
    a1.plot([i+1,i+1], phi_CI[:,i], "r-")
    a1.grid(True)
a1.set_xlim([1/2,len(phi)+1/2])
a2.plot(1, alpha, "o")
a2.plot([1,1], alpha_CI, "r-")
a2.grid(True)
plt.suptitle("Credible intervals");

In [ ]:
fig = plt.figure(figsize=(16,3.5))
plt.subplot(131)
plt.plot(np.arange(1,p+1), beta, "o")
for i in range(p) :
    plt.plot([i+1,i+1], beta_ci_1[i], "r-")
plt.grid(True)
plt.xlabel(r"$\beta$ index", fontsize=12)
plt.ylabel(r"$\beta$", fontsize=12)
plt.title(r"Averaging $m=10$ subposteriors")
plt.text(x=25, y=0.29, s=r"Credible intervals for $\beta$ using divide-and-conquer", fontsize=13)
plt.subplot(132)
plt.plot(np.arange(1,p+1), beta, "o")
for i in range(p) :
    plt.plot([i+1,i+1], beta_ci_2[i], "r-")
plt.grid(True)
plt.yticks(alpha=0)
plt.title(r"Averaging $m=20$ subposteriors")
plt.xlabel(r"$\beta$ index", fontsize=12)
# plt.suptitle(r"Credible intervals for $\beta$ using divide-and-conquer", fontsize=14)
plt.subplot(133)
plt.plot(np.arange(1,p+1), beta, "o")
for i in range(p) :
    plt.plot([i+1,i+1], beta_CI[:,i], "r-")
    plt.grid(True)
plt.title("Credible intervals for full posterior")
plt.yticks(alpha=0)
plt.xlabel(r"$\beta$ index", fontsize=12)
plt.subplots_adjust(wspace=1e-2)
fig.savefig('CI_ar(2)_errors.pdf', bbox_inches='tight', dpi=2000)